# 02 - Data Validation

## Objective
Validate that the data meets expected constraints before any cleaning. This catches data-quality problems early — before they silently corrupt downstream analysis or models.

## Validation checks:
1. **Schema validation** — Do the expected columns exist?
2. **Type validation** — Are key columns the right data type?
3. **Range validation** — Are numerical values within expected bounds?
4. **Uniqueness** — Is the employee identifier unique?
5. **Category validation** — Do categorical columns contain only expected values?

---

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/raw"

# Load all datasets
df_attrition = pd.read_csv(f"{DATA_PATH}/employee_attrition.csv")
df_engagement = pd.read_csv(f"{DATA_PATH}/hr_performance_engagement.csv")
df_occupation = pd.read_csv(f"{DATA_PATH}/occupation_data.csv")
df_essential = pd.read_csv(f"{DATA_PATH}/essential_skills.csv")
df_software = pd.read_csv(f"{DATA_PATH}/software_skills.csv")

print("All 5 datasets loaded.")
print(f"  employee_attrition: {df_attrition.shape}")
print(f"  hr_performance_engagement: {df_engagement.shape}")
print(f"  occupation_data: {df_occupation.shape}")
print(f"  essential_skills: {df_essential.shape}")
print(f"  software_skills: {df_software.shape}")

All 5 datasets loaded.
  employee_attrition: (1470, 35)
  hr_performance_engagement: (2845, 28)
  occupation_data: (1016, 3)
  essential_skills: (18200, 15)
  software_skills: (31821, 7)


---
## 1. Schema Validation
Check that expected important columns exist in each dataset.
We use actual column names discovered during Step 1 — no assumptions.
---

In [2]:
# Schema validation for employee_attrition
expected_attrition_cols = ["Age", "Attrition", "Department", "JobRole",
                          "MonthlyIncome", "EmployeeNumber", "OverTime",
                          "JobSatisfaction", "WorkLifeBalance", "YearsAtCompany"]
actual_cols = set(df_attrition.columns)
missing_cols = [c for c in expected_attrition_cols if c not in actual_cols]

print("=== employee_attrition SCHEMA CHECK ===")
print(f"Expected columns: {len(expected_attrition_cols)}")
print(f"Missing: {missing_cols if missing_cols else 'None'}")
print(f"Result: {'PASS' if not missing_cols else 'FAIL'}")

=== employee_attrition SCHEMA CHECK ===
Expected columns: 10
Missing: None
Result: PASS


In [3]:
# Schema validation for hr_performance_engagement
# Check what columns exist - we adapt to the actual data
expected_engagement_cols = ["Employee ID"]
actual_cols = set(df_engagement.columns)
missing_cols = [c for c in expected_engagement_cols if c not in actual_cols]

print("=== hr_performance_engagement SCHEMA CHECK ===")
print(f"Expected columns: {len(expected_engagement_cols)}")
print(f"Missing: {missing_cols if missing_cols else 'None'}")
print(f"Result: {'PASS' if not missing_cols else 'FAIL'}")

=== hr_performance_engagement SCHEMA CHECK ===
Expected columns: 1
Missing: None
Result: PASS


In [4]:
# Schema validation for occupation_data
expected_occupation_cols = ["O*NET-SOC Code", "Title", "Description"]
actual_cols = set(df_occupation.columns)
missing_cols = [c for c in expected_occupation_cols if c not in actual_cols]

print("=== occupation_data SCHEMA CHECK ===")
print(f"Expected columns: {len(expected_occupation_cols)}")
print(f"Missing: {missing_cols if missing_cols else 'None'}")
print(f"Result: {'PASS' if not missing_cols else 'FAIL'}")

=== occupation_data SCHEMA CHECK ===
Expected columns: 3
Missing: None
Result: PASS


In [5]:
# Schema validation for essential_skills
expected_essential_cols = ["O*NET-SOC Code", "Title", "Element Name"]
actual_cols = set(df_essential.columns)
missing_cols = [c for c in expected_essential_cols if c not in actual_cols]

print("=== essential_skills SCHEMA CHECK ===")
print(f"Expected columns: {len(expected_essential_cols)}")
print(f"Missing: {missing_cols if missing_cols else 'None'}")
print(f"Result: {'PASS' if not missing_cols else 'FAIL'}")

=== essential_skills SCHEMA CHECK ===
Expected columns: 3
Missing: None
Result: PASS


In [6]:
# Schema validation for software_skills
expected_software_cols = ["O*NET-SOC Code", "Title", "Element Name"]
actual_cols = set(df_software.columns)
missing_cols = [c for c in expected_software_cols if c not in actual_cols]

print("=== software_skills SCHEMA CHECK ===")
print(f"Expected columns: {len(expected_software_cols)}")
print(f"Missing: {missing_cols if missing_cols else 'None'}")
print(f"Result: {'PASS' if not missing_cols else 'FAIL'}")

=== software_skills SCHEMA CHECK ===
Expected columns: 3
Missing: None
Result: PASS


---
## 2. Type Validation
Check appropriate data types for key columns.
---

In [7]:
print("=== TYPE VALIDATION ===")
print()

# Employee Attrition - key column types
checks = [
    ("Age", df_attrition["Age"].dtype, "int64", "PASS" if pd.api.types.is_integer_dtype(df_attrition["Age"]) else "FAIL"),
    ("MonthlyIncome", df_attrition["MonthlyIncome"].dtype, "int64/float64", "PASS" if pd.api.types.is_numeric_dtype(df_attrition["MonthlyIncome"]) else "FAIL"),
    ("Attrition", df_attrition["Attrition"].dtype, "text", "PASS" if pd.api.types.is_string_dtype(df_attrition["Attrition"]) else "FAIL"),
    ("Department", df_attrition["Department"].dtype, "text", "PASS" if pd.api.types.is_string_dtype(df_attrition["Department"]) else "FAIL"),
    ("JobRole", df_attrition["JobRole"].dtype, "text", "PASS" if pd.api.types.is_string_dtype(df_attrition["JobRole"]) else "FAIL"),
    ("EmployeeNumber", df_attrition["EmployeeNumber"].dtype, "int64", "PASS" if pd.api.types.is_integer_dtype(df_attrition["EmployeeNumber"]) else "FAIL"),
]

for col_name, actual, expected, result in checks:
    print(f"  {col_name}: {actual} (expected {expected}) -> {result}")

=== TYPE VALIDATION ===

  Age: int64 (expected int64) -> PASS
  MonthlyIncome: int64 (expected int64/float64) -> PASS
  Attrition: str (expected text) -> PASS
  Department: str (expected text) -> PASS
  JobRole: str (expected text) -> PASS
  EmployeeNumber: int64 (expected int64) -> PASS


---
## 3. Range Validation
Check that numerical values fall within expected bounds.
---

In [8]:
print("=== RANGE VALIDATION ===")
print()

# Age should be 18-100
age_min, age_max = df_attrition["Age"].min(), df_attrition["Age"].max()
age_ok = age_min >= 18 and age_max <= 100
print(f"Age range: {age_min} - {age_max}")
print(f"  Expected: 18-100 -> {'PASS' if age_ok else 'FAIL'}")
if not age_ok:
    out_of_range = df_attrition[(df_attrition["Age"] < 18) | (df_attrition["Age"] > 100)]
    print(f"  Out of range rows: {len(out_of_range)}")
print()

# MonthlyIncome should be positive
income_min, income_max = df_attrition["MonthlyIncome"].min(), df_attrition["MonthlyIncome"].max()
income_ok = income_min >= 0
print(f"MonthlyIncome range: {income_min} - {income_max}")
print(f"  Expected: >= 0 -> {'PASS' if income_ok else 'FAIL'}")
print()

# Check for engagement score range if column exists
# The engagement dataset might have different column names
print("Engagement dataset - checking for score columns...")
score_cols = [c for c in df_engagement.columns if "score" in c.lower() or "rating" in c.lower() or "engagement" in c.lower()]
for col in score_cols:
    if pd.api.types.is_numeric_dtype(df_engagement[col]):
        col_min = df_engagement[col].min()
        col_max = df_engagement[col].max()
        print(f"  {col}: range {col_min} - {col_max}")

=== RANGE VALIDATION ===

Age range: 18 - 60
  Expected: 18-100 -> PASS

MonthlyIncome range: 1009 - 19999
  Expected: >= 0 -> PASS

Engagement dataset - checking for score columns...
  Current Employee Rating: range 1 - 5
  Engagement Score: range 1 - 5
  Satisfaction Score: range 1 - 5
  Work-Life Balance Score: range 1 - 5


---
## 4. Uniqueness
Check whether the employee identifier is unique.
---

In [9]:
print("=== UNIQUENESS VALIDATION ===")
print()

# EmployeeNumber in employee_attrition
emp_num_unique = df_attrition["EmployeeNumber"].is_unique
emp_num_dupes = df_attrition["EmployeeNumber"].duplicated().sum()
print(f"EmployeeNumber (employee_attrition):")
print(f"  Unique: {emp_num_unique}")
print(f"  Duplicates: {emp_num_dupes}")
print(f"  Result: {'PASS' if emp_num_unique else 'FAIL'}")
print()

# Employee ID in engagement dataset
if "Employee ID" in df_engagement.columns:
    eid_unique = df_engagement["Employee ID"].is_unique
    eid_dupes = df_engagement["Employee ID"].duplicated().sum()
    print(f"Employee ID (hr_performance_engagement):")
    print(f"  Unique: {eid_unique}")
    print(f"  Duplicates: {eid_dupes}")
    print(f"  Result: {'PASS' if eid_unique else 'FAIL'}")
else:
    print("No 'Employee ID' column found in hr_performance_engagement")
    print("Checking all columns for potential ID fields...")
    for col in df_engagement.columns:
        nunique = df_engagement[col].nunique()
        if nunique == len(df_engagement):
            print(f"  '{col}' has all unique values ({nunique})")

=== UNIQUENESS VALIDATION ===

EmployeeNumber (employee_attrition):
  Unique: True
  Duplicates: 0
  Result: PASS

Employee ID (hr_performance_engagement):
  Unique: True
  Duplicates: 0
  Result: PASS


---
## 5. Category Validation
Verify that categorical columns contain only expected values.
---

In [10]:
print("=== CATEGORY VALIDATION ===")
print()

# Attrition should only be Yes/No
attrition_values = set(df_attrition["Attrition"].dropna().unique())
valid_attrition = {"Yes", "No"}
attrition_ok = attrition_values <= valid_attrition
print(f"Attrition values: {attrition_values}")
print(f"  Expected subset of: {valid_attrition}")
print(f"  Result: {'PASS' if attrition_ok else 'FAIL'}")
if not attrition_ok:
    unexpected = attrition_values - valid_attrition
    print(f"  Unexpected values: {unexpected}")
print()

# Department values
dept_values = sorted(df_attrition["Department"].unique())
print(f"Department values: {dept_values}")
print()

# Gender values
gender_values = sorted(df_attrition["Gender"].unique())
print(f"Gender values: {gender_values}")
print()

# MaritalStatus values
marital_values = sorted(df_attrition["MaritalStatus"].unique())
print(f"MaritalStatus values: {marital_values}")

=== CATEGORY VALIDATION ===

Attrition values: {'Yes', 'No'}
  Expected subset of: {'Yes', 'No'}
  Result: PASS

Department values: ['Human Resources', 'Research & Development', 'Sales']

Gender values: ['Female', 'Male']

MaritalStatus values: ['Divorced', 'Married', 'Single']


---
## Validation Summary

Consolidating all validation results into one table.
---

In [11]:
validation_results = [
    {"Validation": "Required columns (attrition)", "Result": "PASS", "Details": "All expected columns present"},
    {"Validation": "Required columns (engagement)", "Result": "PASS", "Details": "Employee ID column present"},
    {"Validation": "Required columns (occupation)", "Result": "PASS", "Details": "O*NET-SOC Code, Title, Description present"},
    {"Validation": "Required columns (essential skills)", "Result": "PASS", "Details": "O*NET-SOC Code, Element Name present"},
    {"Validation": "Required columns (software skills)", "Result": "PASS", "Details": "O*NET-SOC Code, Element Name present"},
    {"Validation": "Data types", "Result": "PASS", "Details": "Age=integer, MonthlyIncome=numeric, Attrition=object, Department=object"},
    {"Validation": "Age range (18-100)", "Result": "PASS" if age_ok else "FAIL", "Details": f"Actual range: {age_min}-{age_max}"},
    {"Validation": "MonthlyIncome >= 0", "Result": "PASS" if income_ok else "FAIL", "Details": f"Actual range: {income_min}-{income_max}"},
    {"Validation": "EmployeeNumber uniqueness", "Result": "PASS" if emp_num_unique else "FAIL", "Details": f"{emp_num_dupes} duplicates found"},
    {"Validation": "Attrition categories (Yes/No only)", "Result": "PASS" if attrition_ok else "FAIL", "Details": f"Values found: {attrition_values}"},
]

validation_df = pd.DataFrame(validation_results)
validation_df

,Validation,Result,Details
0,Required columns (attrition),PASS,All expected columns present
1,Required columns (engagement),PASS,Employee ID column present
2,Required columns (occupation),PASS,"O*NET-SOC Code, Title, Description present"
3,Required columns (essential skills),PASS,"O*NET-SOC Code, Element Name present"
4,Required columns (software skills),PASS,"O*NET-SOC Code, Element Name present"
5,Data types,PASS,"Age=integer, MonthlyIncome=numeric, Attrition=..."
6,Age range (18-100),PASS,Actual range: 18-60
7,MonthlyIncome >= 0,PASS,Actual range: 1009-19999
8,EmployeeNumber uniqueness,PASS,0 duplicates found
9,Attrition categories (Yes/No only),PASS,"Values found: {'Yes', 'No'}"


---
## Conclusions

- All five datasets have the expected schema.
- Key columns have correct data types.
- Age and MonthlyIncome are within expected ranges.
- EmployeeNumber is unique in employee_attrition.
- Attrition contains only "Yes" and "No" values.
- No blocking validation failures found — data is ready for cleaning.

**Next step:** Data Cleaning (03_data_cleaning.ipynb)